<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z343_MundoIdealEscalado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mundo Ideal Escalado — benchmark sintético de normalizaciones

**Pregunta**: si dos productos tienen la *misma forma* pero distinta escala (uno vende 10x más que el otro),
¿qué normalización logra que se vean como la misma serie?

## Diseño

1. **Shapes base**: 8 formas de 36 meses (tendencia+, tendencia−, estable, estacional, spike, V, escalón, ruido)
2. **Variantes por shape**: K=12 series con escala aleatoria log-uniforme en [0.05, 200]
3. **Normalizaciones**: max, l2, index, reciente, std + sin normalizar
4. **Evaluación**:
   - distancia coseno intra-shape vs inter-shape
   - clustering jerárquico → ARI
   - árbol de decisión supervisado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
T         = 36
K         = 12
HORIZONTE = 2
print('OK')

# 1. Shapes base

In [ ]:
t = np.arange(T)

SHAPES = {
    'tendencia+': lambda: 1 + t/T*3 + np.random.normal(0, 0.05, T),
    'tendencia-': lambda: 4 - t/T*3 + np.random.normal(0, 0.05, T),
    'estable':    lambda: np.ones(T)*2 + np.random.normal(0, 0.08, T),
    'estacional': lambda: 1.5 + np.sin(2*np.pi*t/12) + 0.5*np.sin(2*np.pi*t/6) + np.random.normal(0, 0.05, T),
    'spike':      lambda: np.array([1.2]*T) + np.array([8.0 if i==T//3 else (5.0 if i==2*T//3 else 0) for i in range(T)]) + np.random.normal(0, 0.05, T),
    'V':          lambda: np.concatenate([np.linspace(3, 0.5, T//2), np.linspace(0.5, 3, T-T//2)]) + np.random.normal(0, 0.05, T),
    'escalon':    lambda: np.array([1.0 if i < T//2 else 4.0 for i in range(T)]) + np.random.normal(0, 0.05, T),
    'ruido':      lambda: np.abs(np.random.normal(2, 1.5, T)),
}
N_SHAPES      = len(SHAPES)
shape_nombres = list(SHAPES.keys())

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, (nombre, fn) in enumerate(SHAPES.items()):
    np.random.seed(i)
    axes[i].plot(np.maximum(fn(), 0), 'o-', linewidth=1.5, markersize=3)
    axes[i].set_title(nombre, fontsize=10)
    axes[i].set_ylim(bottom=0)
    axes[i].grid(alpha=0.3)
fig.suptitle('Shapes base (escala=1)', fontsize=12)
plt.tight_layout()
plt.show()

# 2. Generar dataset — mismas formas, escalas aleatorias

In [ ]:
registros = []
rng = np.random.default_rng(99)

for shape_idx, (nombre_shape, fn) in enumerate(SHAPES.items()):
    for k in range(K):
        np.random.seed(shape_idx * 100 + k)
        serie_base   = np.maximum(fn(), 0.0)
        escala_real  = float(np.exp(rng.uniform(np.log(0.05), np.log(200))))
        serie_final  = serie_base * escala_real
        registros.append({
            'id':          f'{nombre_shape}_{k:02d}',
            'shape':       nombre_shape,
            'shape_idx':   shape_idx,
            'k':           k,
            'escala_real': escala_real,
            'serie':       serie_final,
            'serie_base':  serie_base,
        })

labels_reales = np.array([r['shape_idx'] for r in registros])
print(f'{len(registros)} series  |  escalas: {min(r["escala_real"] for r in registros):.3f} – {max(r["escala_real"] for r in registros):.1f}')

# Ver 4 variantes del mismo shape
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, nombre_shape in enumerate(SHAPES):
    ax = axes[i]
    for r in [x for x in registros if x['shape'] == nombre_shape][:4]:
        ax.plot(r['serie'], alpha=0.75, linewidth=1.5, label=f'×{r["escala_real"]:.1f}')
    ax.set_title(nombre_shape, fontsize=9)
    ax.legend(fontsize=6)
    ax.grid(alpha=0.3)
fig.suptitle('4 variantes por shape — misma forma, distinta escala', fontsize=11)
plt.tight_layout()
plt.show()

# 3. Normalizaciones

In [ ]:
def norm_sin(s):      return s.copy(), 1.0

def norm_max(s):
    m = float(s.max()) if s.max() > 0 else 1.0
    return s / m, m

def norm_l2(s):
    n = float(np.sqrt((s**2).sum())) or 1.0
    return s / n, n

def norm_index(s, n_base=3):
    pos  = s[s > 0]
    base = float(pos[:n_base].mean()) if len(pos) > 0 else 1.0
    return s / base, base

def norm_reciente(s, ventana=3):
    base = float(s[-ventana:].mean()) if s[-ventana:].mean() > 0 else 1.0
    return s / base, base

def norm_std(s):
    mu, sigma = float(s.mean()), float(s.std())
    if sigma == 0: sigma = 1.0
    n = (s - mu) / sigma
    if n.min() < 0: n -= n.min()   # desplazar para que mínimo = 0
    return n, (mu, sigma)

NORMALIZACIONES = {
    'sin_norm': norm_sin,
    'max':      norm_max,
    'l2':       norm_l2,
    'index':    norm_index,
    'reciente': norm_reciente,
    'std':      norm_std,
}

# Verificar superposición: 4 variantes del shape estacional
variantes_test = [r for r in registros if r['shape'] == 'estacional'][:4]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()
for i, (nombre_norm, fn) in enumerate(NORMALIZACIONES.items()):
    ax = axes[i]
    for r in variantes_test:
        ax.plot(fn(r['serie'])[0], alpha=0.8, linewidth=1.5)
    ax.set_title(f'norm={nombre_norm}', fontsize=9)
    ax.grid(alpha=0.3)
fig.suptitle('Shape "estacional" — 4 variantes escaladas por normalización\n(si funciona bien, las 4 líneas se superponen)', fontsize=10)
plt.tight_layout()
plt.show()

# 4. Distancia coseno intra-shape vs inter-shape

In [ ]:
def calcular_distancias(registros, norm_fn):
    vectores = [norm_fn(r['serie'])[0] for r in registros]
    labels   = [r['shape_idx'] for r in registros]
    n = len(vectores)
    intra, inter = [], []
    for i in range(n):
        for j in range(i+1, n):
            d = cosine(vectores[i], vectores[j])
            (intra if labels[i]==labels[j] else inter).append(d)
    return np.array(intra), np.array(inter)

resultados_dist = {}
print(f'{"norm":12s}  {"intra":>8s}  {"inter":>8s}  {"sep":>8s}')
print('-' * 42)
for nombre_norm, fn in NORMALIZACIONES.items():
    intra, inter = calcular_distancias(registros, fn)
    sep = inter.mean() - intra.mean()
    resultados_dist[nombre_norm] = {'intra': intra, 'inter': inter, 'sep': sep}
    print(f'{nombre_norm:12s}  {intra.mean():8.4f}  {inter.mean():8.4f}  {sep:8.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, (nombre_norm, res) in enumerate(resultados_dist.items()):
    ax = axes[i]
    ax.hist(res['intra'], bins=40, alpha=0.6, color='steelblue', label='intra-shape (mismo)')
    ax.hist(res['inter'], bins=40, alpha=0.6, color='tomato',    label='inter-shape (distinto)')
    ax.set_title(f'{nombre_norm}  |  sep={res["sep"]:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('distancia coseno')
    ax.grid(alpha=0.3)
fig.suptitle('Distribución de distancias coseno — mayor separación = mejor normalización', fontsize=11)
plt.tight_layout()
plt.show()

# 5. Clustering jerárquico — ARI

In [ ]:
def ari_clustering(registros, norm_fn):
    vectores = [norm_fn(r['serie'])[0] for r in registros]
    Z = linkage(vectores, method='ward')
    labels_pred = fcluster(Z, N_SHAPES, criterion='maxclust')
    return adjusted_rand_score(labels_reales, labels_pred), Z

print('Adjusted Rand Index (1.0 = perfecto, 0.0 = aleatorio):')
print()
ari_resultados = {}
for nombre_norm, fn in NORMALIZACIONES.items():
    ari, Z = ari_clustering(registros, fn)
    ari_resultados[nombre_norm] = (ari, Z)
    print(f'  {nombre_norm:12s}: ARI = {ari:.4f}')

In [ ]:
# Dendrograma mejor vs peor
sorted_ari = sorted(ari_resultados.items(), key=lambda x: x[1][0], reverse=True)
mejor_nombre, (mejor_ari, mejor_Z) = sorted_ari[0]
peor_nombre,  (peor_ari,  peor_Z)  = sorted_ari[-1]

COLORES_SHAPE = plt.cm.tab10(np.linspace(0, 1, N_SHAPES))
labels_plot   = [f"{r['shape'][:4]}_{r['k']:02d}" for r in registros]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, nombre, (ari, Z) in [
    (axes[0], mejor_nombre, (mejor_ari, mejor_Z)),
    (axes[1], peor_nombre,  (peor_ari,  peor_Z)),
]:
    dendrogram(Z, labels=labels_plot, ax=ax, leaf_rotation=90, leaf_font_size=6, color_threshold=0)
    xlbls = ax.get_xmajorticklabels()
    for lbl in xlbls:
        idx = next((i for i, r in enumerate(registros) if f"{r['shape'][:4]}_{r['k']:02d}" == lbl.get_text()), 0)
        lbl.set_color(COLORES_SHAPE[registros[idx]['shape_idx']])
    ax.set_title(f'norm={nombre}  |  ARI={ari:.4f}', fontsize=10)
    ax.set_ylabel('distancia Ward')
    ax.grid(axis='y', alpha=0.3)

handles = [plt.Line2D([0],[0], color=COLORES_SHAPE[i], linewidth=3, label=s) for i, s in enumerate(shape_nombres)]
axes[0].legend(handles=handles, fontsize=7, loc='upper right')
fig.suptitle('Dendrograma — mejor vs peor normalización  (colores = shape real)', fontsize=11)
plt.tight_layout()
plt.show()

# 6. Árbol de decisión — clasificación supervisada de shapes

In [ ]:
print('Accuracy 5-fold CV árbol de decisión (max_depth=5):')
print()
arbol_resultados = {}
for nombre_norm, fn in NORMALIZACIONES.items():
    X = np.array([fn(r['serie'])[0] for r in registros])
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    scores = cross_val_score(clf, X, labels_reales, cv=5, scoring='accuracy')
    arbol_resultados[nombre_norm] = scores
    print(f'  {nombre_norm:12s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Árbol visualizado con la mejor normalización
mejor_norm_arbol = max(arbol_resultados, key=lambda k: arbol_resultados[k].mean())
fn_mejor = NORMALIZACIONES[mejor_norm_arbol]
X = np.array([fn_mejor(r['serie'])[0] for r in registros])
clf = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X, labels_reales)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(clf, ax=ax, class_names=shape_nombres,
          feature_names=[f't{i}' for i in range(T)],
          filled=True, fontsize=7, max_depth=3)
ax.set_title(f'Árbol de decisión — norm={mejor_norm_arbol} (mejor accuracy)', fontsize=11)
plt.tight_layout()
plt.show()

# 7. Caso spike — cómo lo trata cada normalización

In [ ]:
spike_records = [r for r in registros if r['shape'] == 'spike'][:4]
fig, axes = plt.subplots(len(NORMALIZACIONES), len(spike_records),
                          figsize=(14, 3 * len(NORMALIZACIONES)))
for row, (nombre_norm, fn) in enumerate(NORMALIZACIONES.items()):
    for col, r in enumerate(spike_records):
        ax = axes[row, col]
        ax.plot(fn(r['serie'])[0], 'o-', markersize=2, linewidth=1)
        if col == 0:   ax.set_ylabel(nombre_norm, fontsize=8)
        if row == 0:   ax.set_title(f'escala={r["escala_real"]:.1f}', fontsize=8)
        ax.grid(alpha=0.3); ax.tick_params(labelsize=6)

fig.suptitle('Shape "spike" — cómo lo trata cada normalización', fontsize=11)
plt.tight_layout()
plt.show()
print('norm_max:      el spike siempre vale 1.0, el resto queda comprimido')
print('norm_l2:       el spike domina el denominador, efecto similar a max')
print('norm_index:    el spike NO afecta la escala si ocurre después de los primeros 3 meses')
print('norm_reciente: el spike NO afecta la escala si no está en los últimos 3 meses')

# 8. Resumen

In [ ]:
print('=' * 58)
print(f'  {"NORM":12s}  {"Sep coseno":>10s}  {"ARI":>8s}  {"Acc árbol":>10s}')
print('=' * 58)
for nombre_norm in NORMALIZACIONES:
    sep  = resultados_dist[nombre_norm]['sep']
    ari  = ari_resultados[nombre_norm][0]
    acc  = arbol_resultados[nombre_norm].mean()
    print(f'  {nombre_norm:12s}  {sep:>10.4f}  {ari:>8.4f}  {acc:>10.4f}')
print('=' * 58)

# Bar chart ARI
norms   = list(NORMALIZACIONES.keys())
aris    = [ari_resultados[n][0] for n in norms]
colores = ['#2ecc71' if v>0.5 else ('#f39c12' if v>0.25 else '#e74c3c') for v in aris]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, vals, titulo in [
    (axes[0], [resultados_dist[n]['sep'] for n in norms],        'Separación coseno'),
    (axes[1], aris,                                               'ARI clustering'),
    (axes[2], [arbol_resultados[n].mean() for n in norms],       'Accuracy árbol'),
]:
    bars = ax.bar(norms, vals, color=colores, edgecolor='white')
    ax.set_title(titulo, fontsize=10)
    ax.set_xticklabels(norms, rotation=30, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Comparación de normalizaciones — mundo ideal escalado', fontsize=12)
plt.tight_layout()
plt.show()